In [16]:
import ollama

In [17]:
with open('text.tt') as file:
    dataset=file.readlines()
dataset

['The FIFA World Cup is the worldâ€™s premier international football tournament, organized by FIFA every four years, featuring national teams from across the globe. \n',
 'The current champions are Argentina (2022), and the next edition in 2026 will be co-hosted by Canada, Mexico, and the USA with a record 48 teams competing.\n',
 'The FIFA World Cup is not just a sporting eventâ€”itâ€™s a cultural phenomenon. \n',
 'It unites billions of fans worldwide, drives national pride, and showcases footballâ€™s greatest talents on the biggest stage.']

In [18]:
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

In [19]:
vectorDB=[]
def add_chunk(chunk):
    embedding=ollama.embed(model=EMBEDDING_MODEL,input=chunk)['embeddings'][0]
    vectorDB.append((chunk,embedding))

In [20]:
for chunk in dataset:
    add_chunk(chunk)

In [21]:
def cosine_similarity(a,b):
    dotproduct=sum([x*y for x,y in zip(a,b)])
    norm_a=sum([x**2 for x in a])**0.5
    norm_b=sum([y**2 for y in b])**0.5

    return dotproduct/(norm_a*norm_b)

In [22]:
a=[1,2,3]
b=[5,6,7]
eff=cosine_similarity(a,b)
print(eff)

0.9683296637314885


In [23]:
def retrieve(query,top=3):
    embedding=ollama.embed(model=EMBEDDING_MODEL,input=query)['embeddings'][0]
    similarities=[]
    for chunk,emb in vectorDB:
        similarity=cosine_similarity(embedding,emb)
        similarities.append((chunk,similarity))
        similarities.sort(key=lambda x:x[1],reverse=True)
    return similarities[:top]

In [24]:
retrieve('Next World Cup')

[('The FIFA World Cup is the worldâ€™s premier international football tournament, organized by FIFA every four years, featuring national teams from across the globe. \n',
  0.7186952774065808),
 ('The current champions are Argentina (2022), and the next edition in 2026 will be co-hosted by Canada, Mexico, and the USA with a record 48 teams competing.\n',
  0.7131178177141652),
 ('The FIFA World Cup is not just a sporting eventâ€”itâ€™s a cultural phenomenon. \n',
  0.6471369210503999)]

In [25]:
tup=(('anu',12),('manu',11),('vinu',22))
lst=[list(a) for a in tup]
lst.sort(key=lambda x:x[1])

In [26]:
lst

[['manu', 11], ['anu', 12], ['vinu', 22]]

In [27]:
query=input('user:')
retrieve_info=retrieve(query,2)
retrieve_info

[('The current champions are Argentina (2022), and the next edition in 2026 will be co-hosted by Canada, Mexico, and the USA with a record 48 teams competing.\n',
  0.6305502350768537),
 ('The FIFA World Cup is the worldâ€™s premier international football tournament, organized by FIFA every four years, featuring national teams from across the globe. \n',
  0.5207289351188036)]

In [28]:
instructor_prompt=f"""you are a helpful chatbot.
use only the given context of text to answer.
do not create new information for yourself. {''.join(know[0] for know in retrieve_info)}"""

print(instructor_prompt)

you are a helpful chatbot.
use only the given context of text to answer.
do not create new information for yourself. The current champions are Argentina (2022), and the next edition in 2026 will be co-hosted by Canada, Mexico, and the USA with a record 48 teams competing.
The FIFA World Cup is the worldâ€™s premier international football tournament, organized by FIFA every four years, featuring national teams from across the globe. 



In [29]:
stream=ollama.chat(
    model=LANGUAGE_MODEL,
    messages=[{'role':'system','content':instructor_prompt},
              {'role':'user','content':query}],stream=True
)

In [30]:
for i in stream:
    print(i['message']['content'],end='')

Argentina was indeed the champion of the 2022 FIFA World Cup, which took place in Qatar from November 20 to December 18, 2022.